# OPF fine-tune — Phase 3: train + eval (Colab GPU)

Trains the **OpenAI Privacy Filter (OPF)** token-classifier on Leizilla's committed gold
(`data/opf/gold/`) to tag structural markers of Brazilian legislation. This is Phase 3 of
the plan in [`docs/opf-finetune.md`](../docs/opf-finetune.md) (see also ADR-0012 and the
`opf-finetune` skill's `colab-and-drive.md` / `training-and-eval.md`).

**Runtime: GPU.** Set `Runtime → Change runtime type → GPU` (T4 is enough with bf16).
This notebook does **only training/eval** — data prep already happened in the repo; the
gold is the frozen, version-controlled contract.

It is safe to run on the tiny **v0** gold first: that smoke-tests the whole `opf` CLI +
Drive plumbing and gives a v0 baseline F1 *before* investing in a bigger corpus. Expect
undersampled categories (e.g. `ali_marcador` at ~7 train spans) to score F1 ≈ 0 — that is
the model correctly reporting undersampling, **not a bug**.

### Gotchas baked in (from the skill — do not undo)
- **Mount Drive first**, before any install/download, so caching works.
- **Run OPF from the interpreter you installed it into.** Colab has no venv; install with
  `%pip` and invoke with **`sys.executable -m opf`**, never `uv run`.
- **Persist the ~2.8 GB base model to Drive once**; restore from there every later run.
- **Save the checkpoint + metrics to Drive immediately** after training (runtimes are
  ephemeral).
- **Don't cargo-cult `--n-ctx 512`.** On GPU push it up — long context with no chunking is
  OPF's whole advantage. Set as high as VRAM allows.
- **Never run multi-epoch training as a single `opf train` process.** opf's runner
  clones the *entire* model to host RAM every time validation loss improves and holds
  the clone until training ends (`opf/_train/runner.py`, `best_state`). On the standard
  ~13 GB Colab host that is a guaranteed SIGKILL (-9) mid-epoch-2/3 — reproduced twice
  with 3-second RAM traces (2.9 GB steady → 8.3 GB after epoch 1 → 10.6 GB and killed).
  The train cell below therefore runs a **process-per-epoch resume chain**: each epoch
  is its own `opf train --epochs 1` subprocess resumed via `--checkpoint`, so the clone
  never stacks. Validated over 16 consecutive epoch-processes (RAM steady ~1.1 GB
  between epochs).
- **`--batch-size 1` on a T4 — the default (4) OOMs.** With ~10k-token windows,
  batch=1 already uses 12.4/15.4 GB VRAM; batch=2 crashed inside the first forward
  pass. Prefer more epochs at batch 1 over gradient accumulation: with a tiny gold,
  optimizer steps per epoch are the scarce resource (grad-accum 4 cut steps/epoch to
  19 and visibly slowed convergence).
- **`--learning-rate 5e-5`, not the 1e-5 default.** A custom label space rebuilds the
  output head with almost every row randomly initialized (`fallback=…` in the train
  log). Measured head-to-head on the same data/GPU: lr 5e-5 reached val_loss 0.419 in
  2 epochs; lr 1e-5 needed 4 epochs to reach only 0.531.
- **Early epochs will report token accuracy ≈ 92% and span F1 = exactly 0 for every
  category.** That is the all-background ("O") regime every run passes through under
  class imbalance, not a broken model — do not compare that number against the regex
  baseline and conclude OPF is useless. Span predictions lift off only after val_loss
  drops well below ~0.5; the train cell evaluates every 2 epochs to catch the moment.

In [ ]:
# 0. Stdlib imports (once) + GPU check. Fail loudly on a CPU runtime.
import datetime
import json
import os
import shutil
import subprocess
import sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"  # fragmentation guard
)
try:
    GPU_INFO = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    print(GPU_INFO or "nvidia-smi returned no GPU")
except FileNotFoundError:
    GPU_INFO = "nvidia-smi not found (no GPU runtime)"
    print(
        "WARNING: nvidia-smi not found — switch to a GPU runtime "
        "(Runtime > Change runtime type > GPU). CPU works but is slow."
    )

In [ ]:
# 1. Mount Drive at the very top (cache layer for base model + checkpoints).
try:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/opf-finetune"
except Exception:
    ROOT = "/content/opf-finetune"  # non-Colab fallback (no persistence)

os.makedirs(ROOT, exist_ok=True)
print("ROOT =", ROOT)

In [ ]:
# 2. Config — everything that defines a run lives here.

# Where the gold comes from (version-controlled contract). Pin to the merged commit
# for a reproducible run; the branch is fine for iteration.
GOLD_GIT_URL = "https://github.com/franklinbaldo/leizilla.git"
GOLD_GIT_REF = "main"  # or "claude/zealous-noether-nRabQ" until merged

CATEGORY_VERSION = "leizilla_normas_v1"  # must match data/opf/label_space.json
SEED = 13
N_CTX = 8192  # raise toward VRAM limit; v0 laws are < ~3k tokens, headroom is fine
DEVICE = "cuda"

# Training hyperparameters — measured on a T4 against a live opf install
# (2026-07-16, causaganha segmenter run; see PR description for the evidence):
# - BATCH_SIZE 1: the opf default (4) CUDA-OOMs a T4 (batch=1 already ~12.4/15.4 GB
#   with ~10k-token windows; batch=2 died in the first forward pass).
# - LEARNING_RATE 5e-5: the default (1e-5) converges ~4x slower on a rebuilt output
#   head (val_loss 0.419 @ 2 epochs vs 0.531 @ 4 epochs, same data/GPU).
# - EPOCHS as a *chain of single-epoch processes*, never one multi-epoch process —
#   opf keeps a full CPU clone of the best model per improvement and a ~13 GB Colab
#   host gets SIGKILLed mid-epoch-2 (reproduced twice, RAM-traced). See train cell.
EPOCHS = 10
BATCH_SIZE = 1
LEARNING_RATE = "5e-5"
EVAL_EVERY = 2  # span-level eval on val every N epochs — catches span-F1 liftoff


RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

LOCAL_BASE = "/content/base/privacy_filter"
DRIVE_BASE = f"{ROOT}/base/privacy_filter"
DATA_DIR = f"{ROOT}/data/{CATEGORY_VERSION}"  # frozen artifact set
RUN_DIR = f"{ROOT}/checkpoints/{CATEGORY_VERSION}/{RUN_ID}"
OUT_LOCAL = "/content/out/best"  # local train output
for _d in (DATA_DIR, RUN_DIR, "/content/out"):
    os.makedirs(_d, exist_ok=True)
print("RUN_ID =", RUN_ID, "\nDATA_DIR =", DATA_DIR, "\nRUN_DIR =", RUN_DIR)

In [ ]:
# 3. Install the OPF CLI. Colab has no venv -> install into the system interpreter,
#    then ALWAYS invoke via sys.executable (NOT `uv run`).
if not os.path.exists("/content/privacy-filter"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/openai/privacy-filter",
            "/content/privacy-filter",
        ],
        check=True,
    )
PRIVACY_FILTER_COMMIT = subprocess.run(
    ["git", "-C", "/content/privacy-filter", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
# %pip keeps the kernel's interpreter; -e exposes the `opf` CLI (redact/eval/train).
%pip install -q -e /content/privacy-filter
# Sanity: the module must import from THIS interpreter.
subprocess.run(
    [sys.executable, "-c", "import opf, sys; print('opf OK', sys.executable)"],
    check=True,
)

In [ ]:
# Helper: run the opf CLI from the right interpreter and stream output.
def opf(*args, check=True):
    cmd = [sys.executable, "-m", "opf", *map(str, args)]
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)


# Confirm the live flag surface — the skill warns flags beyond
# --validation-dataset/--label-space-json/--output-dir must be verified, not assumed.
opf("train", "--help", check=False)
opf("eval", "--help", check=False)

In [ ]:
# 4. Restore-or-fetch the base checkpoint (~2.8 GB) — download once, persist to Drive.
BASE_MODEL_SHA_FILE = f"{DRIVE_BASE}/.base_model_sha.txt"
if os.path.exists(f"{LOCAL_BASE}/config.json"):
    print("base already local this session")
elif os.path.exists(f"{DRIVE_BASE}/config.json"):
    print("restoring base from Drive…")
    shutil.copytree(DRIVE_BASE, LOCAL_BASE)
else:
    print("first run: downloading base from the Hub, then persisting to Drive…")
    from huggingface_hub import HfApi, snapshot_download

    _sha = HfApi().model_info("openai/privacy-filter").sha
    snapshot_download("openai/privacy-filter", local_dir=LOCAL_BASE, revision=_sha)
    os.makedirs(os.path.dirname(DRIVE_BASE), exist_ok=True)
    shutil.copytree(LOCAL_BASE, DRIVE_BASE)
    with open(BASE_MODEL_SHA_FILE, "w", encoding="utf-8") as fh:
        fh.write(_sha)

# Read back uniformly so BASE_MODEL_SHA is set the same way regardless of which
# branch above ran (fresh download vs. Drive/session cache restore). Fail loudly
# on a legacy Drive cache with no sidecar -- silently recording an unknown
# revision would contradict this manifest's whole point.
if not os.path.exists(BASE_MODEL_SHA_FILE):
    raise RuntimeError(
        "Cached base model has no recorded Hugging Face revision; "
        "delete the legacy cache and download it again."
    )
BASE_MODEL_SHA = open(BASE_MODEL_SHA_FILE, encoding="utf-8").read().strip()
print("base at", LOCAL_BASE, "| base_model_sha =", BASE_MODEL_SHA)

In [ ]:
# 5. Fetch the gold from git (the frozen artifact set) into DATA_DIR.
SRC = "/content/leizilla"
if os.path.exists(SRC):
    shutil.rmtree(SRC)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GOLD_GIT_REF, GOLD_GIT_URL, SRC],
    check=True,
)
SRC_COMMIT = subprocess.run(
    ["git", "-C", SRC, "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
for _f in ("gold/train.jsonl", "gold/val.jsonl", "gold/test.jsonl", "label_space.json"):
    shutil.copy(f"{SRC}/data/opf/{_f}", f"{DATA_DIR}/{os.path.basename(_f)}")
shutil.copy(f"{SRC}/data/opf/gold/manifest.json", f"{DATA_DIR}/gold_manifest.json")
print("gold @", SRC_COMMIT, "->", DATA_DIR)
print(open(f"{DATA_DIR}/gold_manifest.json", encoding="utf-8").read())

In [ ]:
# 6. Validate the gold BEFORE spending a GPU run (offset/overlap/label-space gate).
# Every split's exit code is checked -- a bad val/test split must fail loudly here,
# not silently reach training/eval (opf_annotate.py validate returns 1 on error).
validation_rc = {}
for _split in ("train", "val", "test"):
    validation_rc[_split] = subprocess.run(
        [
            sys.executable,
            f"{SRC}/scripts/opf_annotate.py",
            "validate",
            f"{DATA_DIR}/{_split}.jsonl",
            "--label-space",
            f"{DATA_DIR}/label_space.json",
        ]
    ).returncode
_failed = [s for s, rc in validation_rc.items() if rc != 0]
assert not _failed, f"gold validation failed for: {_failed} -- fix before training"

In [ ]:
# 7. Train — process-per-epoch resume chain (NOT one multi-epoch process).
#
#    Why: opf's runner clones the entire model state_dict to host RAM every time
#    validation loss improves and keeps it until the process exits
#    (opf/_train/runner.py, best_state). Multi-epoch runs on the ~13 GB Colab host
#    get SIGKILLed (-9) mid-epoch-2/3 — reproduced twice with 3s RAM sampling
#    (2.9 GB steady -> 8.3 GB after epoch 1 -> 10.6 GB, killed). One process per
#    epoch resumed via --checkpoint sidesteps it entirely: RAM returns to ~1.1 GB
#    between epochs (validated over 16 consecutive epoch-processes).
#
#    Flags verified against a live `opf train --help` (openai/privacy-filter,
#    2026-07-16) -- re-check if a future opf version errors here.
import json as _json

prev_ckpt = LOCAL_BASE  # epoch 1 starts from the Drive-persisted base (cell 4)
epoch_history = []
for _epoch in range(1, EPOCHS + 1):
    _out = f"/content/out/epoch{_epoch}"
    train_args = [
        "train",
        f"{DATA_DIR}/train.jsonl",
        "--validation-dataset",
        f"{DATA_DIR}/val.jsonl",
        "--label-space-json",
        f"{DATA_DIR}/label_space.json",
        "--output-dir",
        _out,
        "--checkpoint",
        prev_ckpt,
        "--device",
        DEVICE,
        "--n-ctx",
        str(N_CTX),
        "--shuffle-seed",
        str(SEED),
        "--epochs",
        "1",
        "--batch-size",
        str(BATCH_SIZE),
        "--learning-rate",
        str(LEARNING_RATE),
    ]
    opf(*train_args)

    # Per-epoch metrics live in finetune_summary.json under `epoch_metrics`
    # (best_metric / best_epoch are top-level). Key name verified against the
    # written file, not guessed.
    with open(f"{_out}/finetune_summary.json", encoding="utf-8") as _f:
        _summary = _json.load(_f)
    _em = (_summary.get("epoch_metrics") or [{}])[-1]
    epoch_history.append(
        {
            "epoch": _epoch,
            "train_loss": _em.get("train_loss"),
            "val_loss": _em.get("validation_loss"),
            "val_token_accuracy": _em.get("validation_token_accuracy"),
        }
    )
    print(
        f"epoch {_epoch}/{EPOCHS}: "
        f"train_loss={_em.get('train_loss'):.4f} "
        f"val_loss={_em.get('validation_loss'):.4f}"
    )

    # Span-level eval cadence: early epochs WILL show span F1 = 0 for every
    # category while token accuracy sits at ~92% -- the all-background regime,
    # not a bug. This eval-every-N catches the epoch where spans lift off.
    if _epoch % EVAL_EVERY == 0:
        opf(
            "eval",
            f"{DATA_DIR}/val.jsonl",
            "--checkpoint",
            _out,
            "--device",
            DEVICE,
            "--per-class",
            "--metrics-out",
            f"/content/out/val_metrics_e{_epoch}.json",
            check=False,
        )

    # Free the previous epoch's local checkpoint (2.8 GB each; Colab disk is
    # finite). The base on Drive and the latest epoch always survive.
    if prev_ckpt != LOCAL_BASE:
        shutil.rmtree(prev_ckpt, ignore_errors=True)
    prev_ckpt = _out

# Cells 8/9 below expect the final checkpoint at OUT_LOCAL.
shutil.rmtree(OUT_LOCAL, ignore_errors=True)
shutil.move(prev_ckpt, OUT_LOCAL)
print("final checkpoint ->", OUT_LOCAL)
print(_json.dumps(epoch_history, indent=2))


In [ ]:
# 8. Persist checkpoint to Drive IMMEDIATELY (runtime can die at any time).
shutil.copytree(OUT_LOCAL, RUN_DIR, dirs_exist_ok=True)
print("checkpoint saved ->", RUN_DIR)
print("contents:", os.listdir(RUN_DIR))

In [ ]:
# 9. Evaluate on the held-out PT-BR test slice. Look at span-level F1 split into
#    exact vs partial/overlap (boundary drift is OPF's failure mode on formatted text)
#    and the PER-CATEGORY breakdown with support. Capture the output to test_eval.txt.
# Note: `opf eval` takes no --label-space-json -- the checkpoint already encodes the
# label space it was trained with (verified against a live `opf eval --help`, 2026-07-14).
res = subprocess.run(
    [
        sys.executable,
        "-m",
        "opf",
        "eval",
        f"{DATA_DIR}/test.jsonl",
        "--checkpoint",
        RUN_DIR,
    ],
    capture_output=True,
    text=True,
)
print(res.stdout)
print(res.stderr)
with open(f"{RUN_DIR}/test_eval.txt", "w", encoding="utf-8") as fh:
    fh.write(res.stdout + "\n" + res.stderr)
assert res.returncode == 0, "opf eval failed -- see test_eval.txt / stderr above"

### Operating point — bias toward precision (no retrain needed)

OPF decodes with a constrained Viterbi over BIOES tags plus transition-bias parameters
(background persistence, span entry/continuation/closure, boundary handoff). Shifting
those moves the precision/recall operating point **without retraining**:

- discourage background / encourage span entry+continuation → broader spans → ↑ recall;
- the opposite → tighter, fewer spans → ↑ precision.

For legal text a false positive (mislabeling a region) usually costs more than a miss a
reviewer catches, so **bias toward precision** and keep a human-review path. Set the point
on `val`, confirm once on `test`. Exact parameter names: `opf eval --help` and the decode
config in `opf/_core/`.

In [ ]:
# 10. Qualitative check — render predicted spans inline on the test doc and eyeball
#     boundaries (one good render beats a single F1 number for spotting drift).
from transformers import pipeline

clf = pipeline("token-classification", model=RUN_DIR, aggregation_strategy="simple")
test_text = json.loads(open(f"{DATA_DIR}/test.jsonl", encoding="utf-8").readline())[
    "text"
]
for _s in clf(test_text[:4000])[:40]:
    print(
        f"{_s['entity_group']:14s} {_s['start']:5d} {test_text[_s['start'] : _s['end']]!r}"
    )

In [ ]:
# 11. Write the run manifest (reproducibility: ties a checkpoint to exact data,
#     code, weights and environment -- follow-up to PR #110's review, which flagged
#     that the manifest didn't pin the base model/opf/library revisions).
import importlib.metadata as _ilm

import torch as _torch


def _pkg_version(name):
    try:
        return _ilm.version(name)
    except _ilm.PackageNotFoundError:
        return None


run_manifest = {
    "category_version": CATEGORY_VERSION,
    "run_id": RUN_ID,
    "seed": SEED,
    "n_ctx": N_CTX,
    "device": DEVICE,
    "gold_source_commit": SRC_COMMIT,
    "gold_git_ref": GOLD_GIT_REF,
    "base_model": "openai/privacy-filter",
    "base_model_hf_sha": BASE_MODEL_SHA,
    "privacy_filter_commit": PRIVACY_FILTER_COMMIT,
    "opf_version": _pkg_version("opf"),
    "transformers_version": _pkg_version("transformers"),
    "torch_version": _pkg_version("torch"),
    "datasets_version": _pkg_version("datasets"),
    "cuda_version": _torch.version.cuda,
    "gpu_info": GPU_INFO,
    "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open(f"{RUN_DIR}/run_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(run_manifest, fh, ensure_ascii=False, indent=2)
print(json.dumps(run_manifest, ensure_ascii=False, indent=2))
print("\nArtifacts in", RUN_DIR, "->", os.listdir(RUN_DIR))

## Interpreting v0 — and how to scale next

**v0 is a smoke test + baseline, not a deliverable model.** Expect the **global** span
F1 to be exactly 0 in early epochs — with heavy class imbalance the model first
learns to predict background everywhere (token accuracy ≈ 92% while every category's
span F1 is 0). Do **not** read an early-epoch 0 next to the regex baseline's 0.95 as
"OPF lost" — it means undertrained, and the per-2-epoch val evals in the train cell
show when span predictions actually lift off. With ~251 spans over 6
documents, read the result as: does the CLI/Drive chain work end-to-end, and which
categories are starved?

- `art_marcador` / `par_marcador` / `inc_marcador` have enough support (~50–100) to show a
  real signal even at v0.
- `ali_marcador` (~7 train spans), and `ementa`/`vigencia`/`revogacao` (~6 each, one per
  doc) will likely score low/zero F1. **That is the data talking, not a bug** — under ~25
  spans a small classifier can't learn a category and may even drag the dense ones.

**Scaling (Phase 3.5 → v1): add variety, not just volume.** Don't pour in more
Planalto-same-format laws. Target the two gaps this gold has:

1. **Laws dense in alíneas** — CLT, CDC, CPC and other coded statutes — to pull
   `ali_marcador` past ~25 and stop it underfitting every statute that uses alíneas.
2. **Emendas / leis with explicit vigência+revogação** — constitutional amendments and
   parliamentary bills have cleaner, denser `vigencia`/`revogacao` patterns than general
   laws (currently 6 each).

Keep the **Planalto-only** caveat in `manifest.json → known_limitations` exactly as is.
Real cross-fonte stratification (assembleia, casacivil, …) closes when the IA publishes
raw OCR — `opf-sample` is already wired for that and just waiting on the data.

**Sequence:** (this notebook) → v0 smoke test → scale with targeted categories → v1 train
→ compare per-category F1 deltas.